# ThreeOhNine — Mini Emergency Exit Planner

**Problem.** During a fire alarm a person must leave a small building as fast as possible without walking into smoke. The fastest route goes through a corridor with a smoke sensor; the sensor is noisy.

**Three algorithms, one goal** (one from each course group):

1. **A\*** (search) finds the fastest route to the nearest of two exits, and a detour that avoids the corridor.
2. **Bayes' rule** (probabilistic reasoning) turns the sensor reading into P(corridor blocked).
3. **MDP** (sequential decision making) uses the route lengths from A\* and the probability from Bayes to choose **Proceed** or **Detour**.

All probabilities, rewards and the map are toy assumptions (see `planner/scenario.py`).

In [1]:
from planner import scenario as S, astar, update_blocked, build_rewards, bellman_update, plan, report
print('\n'.join(S.MAP))
print('prior P(blocked) =', S.PRIOR_BLOCKED, '| P(alarm|blocked) =', S.P_ALARM_IF_BLOCKED,
      '| P(alarm|clear) =', S.P_ALARM_IF_CLEAR, '| smoke penalty =', S.SMOKE_PENALTY, '| gamma =', S.GAMMA)

. . . . . A
. . # . # .
. . # . # .
S . C . . B
. . # . . .
. . # . . .
prior P(blocked) = 0.2 | P(alarm|blocked) = 0.9 | P(alarm|clear) = 0.15 | smoke penalty = 20 | gamma = 0.9


## 1. A\* — route to the nearest exit

**Input:** 6×6 map, start S, exits A and B, corridor C.  
**Key step:** expand the cell with the lowest f = g + h, where h = Manhattan distance to the nearest exit.  
**Output:** fastest route and a detour that avoids C.

In [2]:
grid, start, exits, sensor = S.parse_map()
goals = list(exits.values())
main = astar(grid, start, goals)
for cell, g, h, f in main['trace']:
    print(f'expand {cell}  g={g}  h={h}  f={f}')
print('fastest:', main['cost'], 'steps ->', main['path'])
detour = astar(grid, start, goals, blocked=[sensor])
print('detour :', detour['cost'], 'steps ->', detour['path'], f"({detour['expanded']} cells expanded)")

expand (3, 0)  g=0  h=5  f=5
expand (3, 1)  g=1  h=4  f=5
expand (3, 2)  g=2  h=3  f=5
expand (3, 3)  g=3  h=2  f=5
expand (3, 4)  g=4  h=1  f=5
expand (3, 5)  g=5  h=0  f=5
fastest: 5 steps -> [(3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5)]
detour : 8 steps -> [(3, 0), (3, 1), (2, 1), (1, 1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5)] (12 cells expanded)


*A\* is optimal because Manhattan distance never overestimates the remaining steps on a 4-connected grid.*

## 2. Bayes — is the corridor blocked?

**Input:** prior P(blocked) = 0.20 and one sensor reading.  
**Key step:** P(B|o) = P(o|B)·P(B) / [P(o|B)·P(B) + P(o|¬B)·P(¬B)].  
**Output:** posterior probability that the corridor is blocked.

In [3]:
for alarm in (True, False):
    b = update_blocked(S.PRIOR_BLOCKED, S.P_ALARM_IF_BLOCKED, S.P_ALARM_IF_CLEAR, alarm)
    print(f"{b['observation']:>8}: {b['joint_blocked']:.4f} / ({b['joint_blocked']:.4f} + {b['joint_clear']:.4f}) = {b['posterior']:.4f}")

   alarm: 0.1800 / (0.1800 + 0.1200) = 0.6000
no alarm: 0.0200 / (0.0200 + 0.6800) = 0.0286


*One noisy reading moves the belief from 20% to 60% (alarm) or down to 2.9% (no alarm).*

## 3. MDP — Proceed or Detour?

**Input:** states {Clear, Blocked}, actions {Proceed, Detour}, rewards from the A\* route lengths, transition table, γ = 0.9, belief from Bayes.  
**Key step:** one Bellman update Q(s,a) = R(s,a) + γ Σ T(s'|s,a) V0(s'), then Q(b,a) = Σ b(s) Q(s,a).  
**Output:** the action with the higher expected value.

In [4]:
rewards = build_rewards(main['cost'], detour['cost'], main['path'].index(sensor), S.SMOKE_PENALTY)
print('R =', rewards)
for alarm in (True, False):
    p = update_blocked(S.PRIOR_BLOCKED, S.P_ALARM_IF_BLOCKED, S.P_ALARM_IF_CLEAR, alarm)['posterior']
    m = bellman_update(rewards, S.TRANSITIONS, S.GAMMA, p)
    print(f"P(blocked)={p:.4f}: Q(Proceed)={m['q_belief']['Proceed']:.3f}  Q(Detour)={m['q_belief']['Detour']:.3f}  -> {m['best']}")
print('threshold: detour when P(blocked) >', round(m['threshold'], 4))

R = {('Clear', 'Proceed'): -5, ('Blocked', 'Proceed'): -32, ('Clear', 'Detour'): -8, ('Blocked', 'Detour'): -8}
P(blocked)=0.6000: Q(Proceed)=-27.428  Q(Detour)=-12.635  -> Detour
P(blocked)=0.0286: Q(Proceed)=-10.611  Q(Detour)=-12.635  -> Proceed
threshold: detour when P(blocked) > 0.0973


*The MDP detours only when the expected cost of meeting smoke outweighs the 3 extra steps of the detour.*

## 4. The whole pipeline

In [5]:
print(report(plan(alarm=True)))

1) A* SEARCH - route to the nearest exit
   Input : 6x6 map, start (3, 0), exits Exit A (0, 5), Exit B (3, 5), sensor corridor (3, 2)
   Steps : f = g + h, h = Manhattan distance to the nearest exit
           expand (3, 0)  g=0  h=5  f=5
           expand (3, 1)  g=1  h=4  f=5
           expand (3, 2)  g=2  h=3  f=5
           expand (3, 3)  g=3  h=2  f=5
           expand (3, 4)  g=4  h=1  f=5
           expand (3, 5)  g=5  h=0  f=5
   Output: fastest route to Exit B, 5 steps: (3,0) -> (3,1) -> (3,2) -> (3,3) -> (3,4) -> (3,5)
           detour avoiding (3, 2) to Exit A, 8 steps (12 cells expanded): (3,0) -> (3,1) -> (2,1) -> (1,1) -> (0,1) -> (0,2) -> (0,3) -> (0,4) -> (0,5)
   Why   : A* is optimal with an admissible heuristic and expands fewer cells than blind search.

2) BAYESIAN INFERENCE - is the corridor blocked?
   Input : prior P(blocked) = 0.20, P(alarm|blocked) = 0.90, P(alarm|clear) = 0.15, reading = alarm
   Steps : P(o|blocked) * P(blocked) = 0.90 * 0.20 = 0.1800
      

## 5. Evaluation (100,000 simulated drills)

In [6]:
import runpy
_ = runpy.run_path('scripts/evaluate.py')

Policy comparison (100,000 simulated drills)
  A* only (always fastest route)       avg steps  6.405   walked into smoke 20.08%
  Always detour                        avg steps    8.0   walked into smoke   0.0%
  A* + MDP, no sensor (prior only)     avg steps    8.0   walked into smoke   0.0%
  Full planner (A* + Bayes + MDP)      avg steps  6.041   walked into smoke  1.97%

Sensitivity (detour when P(blocked) > 0.0973)
  prior 0.05: alarm -> 0.2400 Detour   no alarm -> 0.0062 Proceed
  prior 0.10: alarm -> 0.4000 Detour   no alarm -> 0.0129 Proceed
  prior 0.20: alarm -> 0.6000 Detour   no alarm -> 0.0286 Proceed
  prior 0.40: alarm -> 0.8000 Detour   no alarm -> 0.0727 Proceed

Search effort over 28 start cells: A* 4.54 vs BFS 13.5 expanded cells on average (66.4% fewer); scenario start: A* 6 vs BFS 22
